# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

**TTS source:** Bryce Beattie's Piper Clean 100, trained from scratch on LibriTTS-R train-clean-100 (CC BY 4.0). The run archive includes the source, license and SHA-256 of the exact weights and configuration. [Author release](https://brycebeattie.com/files/tts/) · [LibriTTS-R](https://www.openslr.org/141/).

Data preparation downloads approximately 15 GB of archives; allow at least 50 GB of free disk space. It requires the system `zip` command for the multipart FSD50K archive.

In [27]:
# 路径和唤醒词名称集中设置；修改后请从本单元格开始重新运行。
from pathlib import Path
import os

repo_path = Path("/content/micro-wake-word").expanduser().resolve()
data_dir = repo_path / "data_v1"
wake_word_name = "hi_fondcircle"
wake_word_phonemes = "hˈaɪ fˈɒnd sˈɜːkəl"
hey_hi_phonemes = "hˈeɪ hˈaɪ"
tts_version = "clean100_v1"
positive_dataset_name = f"{tts_version}_three_speeds"
sample_root = repo_path / "generated_samples" / wake_word_name
preview_dir = sample_root / f"{positive_dataset_name}_preview"
sample_dir = sample_root / positive_dataset_name
clean100_model_dir = data_dir / "clean100_model"
features_dir = repo_path / "features_v1"
positive_dir = features_dir / f"positive_{wake_word_name}_{positive_dataset_name}"
hey_hi_dataset_version = "clean100_v1"
# 可指向持久存储中已有的 raw 目录，保留 WAV 和 generation.json。
hey_hi_audio_dir = repo_path / "generated_datasets" / f"hey_hi_negative_{hey_hi_dataset_version}" / "raw"
hey_hi_features_dir = features_dir / "negative" / f"hey_hi_{hey_hi_dataset_version}"
hey_hi_context_frames = 204  # 与下方 mixednet 结构及 clip_duration_ms=1500 配套
run_suffix = f"_{tts_version}_hey_hi"
train_dir = repo_path / "trained_models" / f"{wake_word_name}_licensed_inputs{run_suffix}"
config_path = repo_path / f"training_parameters_{wake_word_name}{run_suffix}.yaml"
archive_path = repo_path.parent / f"{wake_word_name}{run_suffix}_run.tar.gz"
data_root = data_dir / "prepared"

# CHiME-6 真实远场 ambient 测试集：prepared/ 复用优先；压缩包与其它数据集放在同一目录。
chime6_dir = repo_path / "chime6_v1"  # 与命令行 --output-dir 默认值一致
chime6_features_dir = features_dir / "negative" / "ambient_chime6"

# Colab 首次运行时获取仓库。
import subprocess
if not repo_path.exists():
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/Seeker32/micro-wake-word.git", str(repo_path)], check=True)
os.chdir(repo_path)
assert (repo_path / "microwakeword/audio/licensed_data.py").is_file(), f"找不到仓库：{repo_path}"

if not (repo_path / "microwakeword/audio/piper_clean100.py").is_file():
    raise FileNotFoundError("请将仓库更新到包含 microwakeword/audio/piper_clean100.py 的版本。")


In [10]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
import platform
from pathlib import Path

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

%pip install -e "{repo_path}"

Obtaining file:///content/micro-wake-word
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for microwakeword (pyproject.toml) ... done
  Created wheel for microwakeword: filename=microwakeword-0.1.0-0.editable-py3-none-any.whl size=9949 sha256=118bf0fb9513d4a4db1039ffe9857ea8abdada52361e5d1fb34ec76273cdb6cd
  Stored in directory: /tmp/pip-ephem-wheel-cache-fkfmilco/wheels/37/04/57/bffe1d22e8e88eedae5f8d49ca012354573c3c6f545cede40b
Successfully built microwakeword
  Attempting uninstall: microwakeword
    Found existing installation: microwakeword 0.1.0
    Uninstalling microwakeword-0.1.0:
      Successfully uninstalled microwakeword-0.1.0


In [28]:
import importlib.util
import subprocess
import sys

from IPython.display import Audio
from microwakeword.audio.piper_clean100 import (
    load_clean100, prepare_clean100_samples, write_clean100_preview,
)

if importlib.util.find_spec("pip") is None:
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "piper-tts==1.3.0"], check=True)

clean100_voice = load_clean100(clean100_model_dir)
sample_path = preview_dir / "0.wav"
write_clean100_preview(clean100_voice, sample_path, wake_word_phonemes)
Audio(filename=str(sample_path), autoplay=True)


Clean 100 · 推理设备：CPUExecutionProvider


In [29]:
# 每档 1000 条正样本。每条使用一个 Clean 100 说话人 ID。
counts = prepare_clean100_samples(clean100_voice, sample_dir, wake_word_phonemes)
print(f"原始正样本：{counts}，共 {sum(counts.values())} 条。")
print(f"试听各档：{sample_dir}/{{fast,normal,slow}}/0.wav")


clean100_v1_three_speeds · fast 已生成 1000 条 WAV
clean100_v1_three_speeds · normal 已生成 1000 条 WAV
clean100_v1_three_speeds · slow 已生成 1000 条 WAV
原始正样本：{'fast': 1000, 'normal': 1000, 'slow': 1000}，共 3000 条。
试听各档：/content/micro-wake-word/generated_samples/hi_fondcircle/clean100_v1_three_speeds/{fast,normal,slow}/0.wav


### 首次生成或复用 hey hi 困难负样本

首次运行时，下一单元格用 Clean 100 合成快、正常、慢三档 `hey hi`，每档 1,000 条，并逐档按 80/10/10 划分。之后运行会校验并复用 `hey_hi_audio_dir` 下的 WAV 和 `generation.json`。

跨会话或跨 notebook 复用时，保留整个 `raw/` 目录，并在顶部把 `hey_hi_audio_dir` 指向同一份持久目录。特征也会在 `hey_hi_features_dir` 下校验后复用。修改 `hey_hi_phonemes` 时，请更新 `hey_hi_dataset_version` 和 `tts_version`，为新数据创建独立的缓存和训练目录。


In [30]:
# 首次生成 hey hi，已有数据只校验和复用。
reuse_hey_hi = hey_hi_audio_dir.is_dir()
# 与下方特征导出使用同样的 10 ms 步长和 0.2 秒尾部留白。
max_duration_s = ((hey_hi_context_frames - 1) * 10 - 200) / 1000
negative_counts = prepare_clean100_samples(
    clean100_voice, hey_hi_audio_dir, hey_hi_phonemes,
    hard_negative_max_duration_s=max_duration_s,
)
action = "复用" if reuse_hey_hi else "生成"
print(f"已{action} Clean 100 hey hi 困难负样本：{negative_counts}")


已复用 Clean 100 hey hi 困难负样本：{'training': 2400, 'validation': 300, 'testing': 300}


In [31]:
# 读取原有划分的 hey hi WAV，并在训练端生成特征。
from microwakeword.audio.hard_negatives import prepare_hard_negative_features

hey_hi_lineage = prepare_hard_negative_features(
    hey_hi_audio_dir, hey_hi_features_dir,
    context_frames=hey_hi_context_frames,
)
print("hey hi 各集合特征数量:", hey_hi_lineage["counts"])


hey hi 各集合特征数量: {'training': 2400, 'validation': 300, 'testing': 300}


In [32]:
# （可选）从 Google Drive 预置数据集压缩包，替代下一单元格约 15 GB 的原始下载。
# 原理：prepare_licensed_data() 把压缩包缓存在 data_dir/downloads，download() 见到同名
# 文件就跳过下载（随后按发布版 MD5 校验）。因此只要压缩包与官方下载逐字节一致并保持
# 原文件名，复制进缓存目录即可跳过 Zenodo/OpenSLR 的慢速下载。
# 需要在 DRIVE_ARCHIVE_DIR 指向的 Drive 文件夹里放好以下 8 个压缩包（文件名不变）：
#   FSD50K.eval_audio.zip 与 FSD50K.eval_audio.z01（同一分卷的两部分，缺一不可）、
#   FSD50K.metadata.zip、FSD50K.doc.zip、rirs_noises.zip、
#   train-clean-100.tar.gz、dev-clean.tar.gz、test-clean.tar.gz
# 说明：
# - FSD50K 与 LibriSpeech 有官方 MD5，复制完立即核对；不一致就删除坏副本并报错，
#   避免下一单元格在解压阶段才失败。rirs_noises.zip 官方未发布 MD5，
#   由下一单元格 zipfile 的完整性检查兜底。
# - 缓存里已有的文件跳过复制，也不重复哈希（下一单元格本来就会校验）。
# - 非 Colab 环境运行时，把 DRIVE_ARCHIVE_DIR 改成本地镜像目录即可，其余逻辑不变。

import hashlib
import shutil

from microwakeword.audio.licensed_data import FSD_FILES, SPEECH_FILES

DRIVE_ARCHIVE_DIR = Path("/content/drive/MyDrive/wake-word/data_v1/downloads")

try:
    from google.colab import drive
    in_colab = True
except ModuleNotFoundError:
    in_colab = False

if in_colab:
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    archive_source = DRIVE_ARCHIVE_DIR
else:
    archive_source = DRIVE_ARCHIVE_DIR.expanduser()
    print(f"非 Colab 环境：从本地镜像目录读取 {archive_source}")

published_md5 = dict(FSD_FILES)
published_md5.update({f"{subset}.tar.gz": md5 for _, (subset, md5) in SPEECH_FILES.items()})
expected = sorted({*published_md5, "rirs_noises.zip"})

cache_dir = data_dir / "downloads"
cache_dir.mkdir(parents=True, exist_ok=True)

copied = []
for name in expected:
    target = cache_dir / name
    if target.is_file():
        print(f"已存在，跳过复制：{name}")
        continue
    source = archive_source / name
    if not source.is_file():
        raise FileNotFoundError(
            f"{archive_source} 缺少 {name}。请先上传该压缩包（保持原文件名），"
            "或跳过本单元格，让下一单元格直接从官方源下载。"
        )
    gib = source.stat().st_size / 1024 ** 3
    print(f"复制 {name}（{gib:.2f} GiB）...", flush=True)
    shutil.copyfile(source, target)
    copied.append(name)

def _md5_of(path):
    result = hashlib.md5()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            result.update(chunk)
    return result.hexdigest()

for name in copied:
    if name not in published_md5:
        continue  # rirs_noises.zip 无官方 MD5，跳过
    actual = _md5_of(cache_dir / name)
    if actual != published_md5[name]:
        (cache_dir / name).unlink()
        raise ValueError(f"{name} 的 MD5 与发布值不符（{actual}），坏副本已删除。可重试或改走直连下载。")

total_gib = sum((cache_dir / n).stat().st_size for n in expected) / 1024 ** 3
print(f"压缩包就绪：{cache_dir}（本次复制 {len(copied)} 个，缓存共 {total_gib:.2f} GiB）。")


已存在，跳过复制：FSD50K.doc.zip
已存在，跳过复制：FSD50K.eval_audio.z01
已存在，跳过复制：FSD50K.eval_audio.zip
已存在，跳过复制：FSD50K.metadata.zip
已存在，跳过复制：dev-clean.tar.gz
已存在，跳过复制：rirs_noises.zip
已存在，跳过复制：test-clean.tar.gz
已存在，跳过复制：train-clean-100.tar.gz
压缩包就绪：/content/micro-wake-word/data_v1/downloads（本次复制 0 个，缓存共 13.64 GiB）。


In [33]:
# Prepare replacement augmentation and negative audio, with per-file provenance.
# First run downloads ~15 GB; later runs verify the prepared audio hashes.
# Only CC0/CC-BY FSD50K clips and simulated OpenSLR 28 RIRs are selected.
# LibriSpeech supplies the replacement speech negatives.
# TTS provenance and product dependencies remain outside this data audit.

from microwakeword.audio.licensed_data import prepare_licensed_data

data_root = prepare_licensed_data(str(data_dir))
print(f"Audio: {data_root}")
print(f"License manifest: {data_root / 'manifest.csv'}")
print(f"Attribution: {data_root / 'ATTRIBUTION.md'}")


Audio: /content/micro-wake-word/data_v1/prepared
License manifest: /content/micro-wake-word/data_v1/prepared/manifest.csv
Attribution: /content/micro-wake-word/data_v1/prepared/ATTRIBUTION.md


In [34]:
# Sets up the augmentations.
# To improve your model, experiment with these settings and use more sources of
# background clips.

from pathlib import Path

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

if not (data_root / "manifest.json").is_file():
    raise FileNotFoundError(
        f"No prepared data at {data_root.resolve()}. "
        "Run the data preparation cell from the repository directory first."
    )

clips = Clips(input_directory=str(sample_dir),
              file_pattern='*/*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
print(f"已载入 {len(clips.clips)} 条原始正样本。")
# Separate background pools keep held-out audio out of training augmentation.
augmenters = {
    split: Augmentation(
        augmentation_duration_s=3.2,
        augmentation_probabilities={
            "SevenBandParametricEQ": 0.1,
            "TanhDistortion": 0.1,
            "PitchShift": 0.1,
            "BandStopFilter": 0.1,
            "AddColorNoise": 0.1,
            "AddBackgroundNoise": 0.75,
            "Gain": 1.0,
            "RIR": 0.5,
        },
        impulse_paths=[str(data_root / "rirs")],
        background_paths=[str(data_root / "noise" / split)],
        background_min_snr_db=-5,
        background_max_snr_db=10,
        min_jitter_s=0.195,
        max_jitter_s=0.205,
    )
    for split in ("training", "validation", "testing")
}
augmenter = augmenters["training"]


已载入 3000 条原始正样本。


In [35]:
# Augment a random clip and play it back to verify it works well

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, str(repo_path / "augmented_clip.wav"))

Audio(str(repo_path / "augmented_clip.wav"), autoplay=True)

In [36]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = str(positive_dir)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,    # Window length is 30 ms
                                     augmenter=augmenters[split],
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenters[split],
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [37]:
# Build replacement negative features from the audited raw audio.
# Keep every source split intact. Ambient evaluation uses original clips >=6s,
# without synthetic repetition; it is a proxy for real-device room recordings.

import json
from pathlib import Path
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.licensed_data import digest

negative_root = features_dir / "negative"
for corpus, pattern in (("speech", "*.flac"), ("noise", "*.wav")):
    for split in ("training", "validation", "testing"):
        source_dir = data_root / corpus / split
        negative_clips = Clips(
            input_directory=str(source_dir),
            file_pattern=pattern,
            repeat_clip_min_duration_s=6,
        )
        generator = SpectrogramGeneration(
            clips=negative_clips,
            step_ms=10,
            split_spectrogram_duration_s=5,
        )
        out_dir = negative_root / corpus / split
        out_dir.mkdir(parents=True, exist_ok=True)
        RaggedMmap.from_generator(
            out_dir=str(out_dir / "negative_mmap"),
            sample_generator=generator.spectrogram_generator(),
            batch_size=100,
            verbose=True,
        )
        if split != "training":
            ambient_clips = Clips(
                input_directory=str(source_dir),
                file_pattern=pattern,
                min_clip_duration_s=6,
            )
            if len(ambient_clips.clips) == 0:
                raise ValueError(f"No original clips >=6s for ambient evaluation: {source_dir}")
            ambient_generator = SpectrogramGeneration(clips=ambient_clips, step_ms=10)
            out_dir = negative_root / "ambient" / f"{split}_ambient"
            out_dir.mkdir(parents=True, exist_ok=True)
            RaggedMmap.from_generator(
                out_dir=str(out_dir / f"{corpus}_mmap"),
                sample_generator=ambient_generator.spectrogram_generator(),
                batch_size=100,
                verbose=True,
            )

# Link derived features to the exact original-file license manifest.
lineage = {
    "audio_manifest": str((data_root / "manifest.json").resolve()),
    "audio_manifest_sha256": digest(data_root / "manifest.json"),
    "negative_features": str(negative_root),
    "positive_features": str(positive_dir),
    "positive_generation_manifest": str(sample_dir / "generation.json"),
    "processing": "16 kHz mono on access; 10 ms feature step; 5s negative chunks; "
                  "short classification negatives repeated to >=6s; ambient clips not repeated",
    "positive_augmentation": {split: {
        "background_pool": str(data_root / "noise" / split),
        "rir_pool": str(data_root / "rirs"),
    } for split in ("training", "validation", "testing")},
    "pending": "Dependency audit and real-device evaluation",
}
lineage["hard_negatives"] = {"hey_hi": hey_hi_lineage}
(negative_root.parent / "feature_sources.json").write_text(json.dumps(lineage, indent=2) + "\n")


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

1003028

### CHiME-6 真实远场 ambient 评估集

误唤醒只有在真实房间录音上量才有意义：FSD50K 的噪声片段很干净，LibriSpeech 是近讲朗读，两者都会低估家庭场景。这里接入 CHiME-6（OpenSLR SLR150，CC BY-SA 4.0）——与 DiPCo/CHiME-5 同源的家庭多人晚餐对话，取每个房间参考阵列的 CH1，切成 10 分钟连续片段。

- 只做 ambient 评估：`sampling_weight = 0`，不参与训练采样，也不做重复或切块，保持原始连续音频。
- dev → `validation_ambient`（4.46 h），eval → `testing_ambient`（15.62 h），沿用官方 session 划分，不跨 split。
- 数据来源优先级：`chime6_v1/prepared`（校验通过即复用）→ 与其它数据集同一目录里的 `chime6_v1.zip`（`data_v1/downloads`，或 Drive 上那个下载目录；解压后是 `prepared/`，也接受已解压目录）→ OpenSLR 流式下载（可断点续传，约 11.9 GB，峰值磁盘约 12 GB，成功后只留 prepared）。
- 内存：`validation_ambient` 会在训练时整体载入（4.46 h ≈ 1.2 GB）。内存吃紧时可改成 `prepare_ambient_features(..., splits=("testing",))` 只建 testing 侧，训练期的 `validation_ambient` 就回落到 FSD50K/LibriSpeech。


In [ ]:
# 接入 CHiME-6：复用 prepared/ → 同目录的 chime6_v1.zip → 才从 OpenSLR 流式下载（断点续传）。
# 只构建 ambient 特征（validation_ambient / testing_ambient），不参与训练采样。
# 压缩包不用单独配置路径：与其它数据集一样放进 data_v1/downloads（或 Drive 的同一个下载目录）。
from microwakeword.audio.chime6 import ensure_chime6, prepare_ambient_features

chime6_source_dirs = [data_dir / "downloads"]
if "DRIVE_ARCHIVE_DIR" in globals():  # 上面的 Drive 单元格已挂载并定义该目录
    chime6_source_dirs.append(DRIVE_ARCHIVE_DIR)
chime6_prepared_dir = ensure_chime6(chime6_dir, source_dirs=chime6_source_dirs)
chime6_lineage = prepare_ambient_features(chime6_prepared_dir, chime6_features_dir)
print(f"CHiME-6 特征目录：{chime6_features_dir}")
print(f"片段数 {len(chime6_lineage['clips'])}，时长(小时)："
      f"{ {split: round(hours, 2) for split, hours in chime6_lineage['hours'].items()} }")


In [38]:
# Save a yaml config that controls the training process
# These hyperparamters can make a huge different in model quality.
# Experiment with sampling and penalty weights and increasing the number of
# training steps.

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = str(train_dir)


# Each feature_dir should have at least one of the following folders with this structure:
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight: Weight for choosing a spectrogram from this set in the batch
#  penalty_weight: Penalizing weight for incorrect predictions from this set
#  truth: Boolean whether this set has positive samples or negative samples
#  truncation_strategy = If spectrograms in the set are longer than necessary for training, how are they truncated
#       - random: choose a random portion of the entire spectrogram - useful for long negative samples
#       - truncate_start: remove the start of the spectrogram
#       - truncate_end: remove the end of the spectrogram
#       - split: Split the longer spectrogram into separate spectrograms offset by 100 ms. Only for ambient sets

config["features"] = [
    {
        "features_dir": str(positive_dir),
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": str(features_dir / "negative" / "speech"),
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": str(features_dir / "negative" / "noise"),
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    { # Only used for validation and testing
        "features_dir": str(features_dir / "negative" / "ambient"),
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]



# 独立采样 hey hi 困难负样本，保留普通语音/噪声负样本。
config["features"].insert(3, {
    "features_dir": str(hey_hi_features_dir),
    "sampling_weight": 2.0,
    "penalty_weight": 1.0,
    "truth": False,
    "truncation_strategy": "truncate_start",  # 短语在末尾，避免随机采到静音
    "type": "mmap",
})

# Penalizing weight for incorrect class predictions - lists that correspond to training steps
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

# CHiME-6 真实远场家庭录音：只进 validation_ambient / testing_ambient，
# sampling_weight=0 表示不参与训练采样；它的误唤醒率会并入 ambient 汇总指标，
# CHiME-6 单独的数字见后面的 evaluate_ambient_model 单元格。
config["features"].append({
    "features_dir": str(chime6_features_dir),
    "sampling_weight": 0.0,
    "penalty_weight": 1.0,
    "truth": False,
    "truncation_strategy": "split",
    "type": "mmap",
})

# Number of training steps in each iteration - various other settings are configured as lists that corresponds to different steps
config["training_steps"] = [10000]
config["learning_rates"] = [0.001]  # Learning rates for Adam optimizer - list that corresponds to training steps
config["batch_size"] = 128

config["time_mask_max_size"] = [0]  # SpecAugment - list that corresponds to training steps
config["time_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_max_size"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps

config["eval_step_interval"] = (500)  # Test the validation sets after every this many stepsz
config["clip_duration_ms"] = (1500)  # Maximum length of wake word that the streaming model will accept


# The best model weights are chosen first by minimizing the specified minimization metric below the specified target_minimization
# Once the target has been met, it chooses the maximum of the maximization metric. Set 'minimization_metric' to None to only maximize
# Available metrics:
#   - "loss" - cross entropy error on validation set
#   - "accuracy" - accuracy of validation set
#   - "recall" - recall of validation set
#   - "precision" - precision of validation set
#   - "false_positive_rate" - false positive rate of validation set
#   - "false_negative_rate" - false negative rate of validation set
#   - "ambient_false_positives" - count of false positives from the split validation_ambient set
#   - "ambient_false_positives_per_hour" - estimated number of false positives per hour on the split validation_ambient set
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # Set to None to disable

config["maximization_metric"] = "average_viable_recall"

# 当前命令的卷积结构共丢弃 154 帧；模型结构改变时需同步检查此值。
expected_context = 1 + (config["clip_duration_ms"] - 30) // (3 * config["window_step_ms"]) + 154
if hey_hi_context_frames != expected_context:
    raise ValueError(f"hey_hi_context_frames 应与训练上下文一致: {expected_context}")

with open(config_path, "w") as file:
    documents = yaml.dump(config, file)

In [39]:
# Trains a model. When finished, it will quantize and convert the model to a
# streaming version suitable for on-device detection.
# It will resume if stopped, but it will start over at the configured training
# steps in the yaml file.
# Change --train 0 to only convert and test the best-weighted model.
# On Google colab, it doesn't print the mini-batch results, so it may appear
# stuck for several minutes! Additionally, it is very slow compared to training
# on a local GPU.

!python -m microwakeword.model_train_eval \
--training_config="{config_path}" \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

INFO:absl:Loading and analyzing data sets.
W0000 00:00:1794374486.834165   46182 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (128, 204, 40)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (128, 204, 1, 40) │          0 │ input_layer[0][0] │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stream (Stream)     │ (128, 67, 1, 32)  │      6,

In [40]:
# 单独评估 hey hi 测试集：每条音频触发与否，不与 ambient 的每小时误唤醒率混算。
import json
from microwakeword.audio.hard_negatives import evaluate_hard_negative_model

hey_hi_report = evaluate_hard_negative_model(
    train_dir / "tflite_stream_state_internal_quant/stream_state_internal_quant.tflite",
    hey_hi_features_dir,
)
report_path = train_dir / "hey_hi_test_results.json"
report_path.write_text(json.dumps(hey_hi_report, indent=2) + "\n", encoding="utf-8")
for result in hey_hi_report["results"]:
    print(f"hey hi 阈值 {result['cutoff']:.2f}: "
          f"误触发 {result['false_triggers']}/{hey_hi_report['sample_count']} "
          f"({result['false_trigger_rate']:.2%})")


hey hi 阈值 0.95: 误触发 0/300 (0.00%)
hey hi 阈值 0.98: 误触发 0/300 (0.00%)
hey hi 阈值 0.99: 误触发 0/300 (0.00%)


In [ ]:
# CHiME-6 ambient 误唤醒报告：每小时误唤醒次数（与 testing_ambient 同口径），
# 并列出触发最强的片段，便于判断是电视/音乐还是家庭对话造成的误唤醒。
import json
from pathlib import Path

import numpy as np

from microwakeword.audio.chime6 import evaluate_ambient_model

ambient_model_path = (train_dir / "tflite_stream_state_internal_quant"
                      / "stream_state_internal_quant.tflite")
ambient_cutoffs = np.array([0.9, 0.95, 0.98, 0.99])
chime6_report = {
    split: evaluate_ambient_model(ambient_model_path, chime6_features_dir,
                                  split=split, cutoffs=ambient_cutoffs)
    for split in sorted(chime6_lineage["hours"], reverse=True)  # testing 优先
}
report_path = train_dir / "chime6_ambient_results.json"
report_path.write_text(json.dumps(chime6_report, indent=2) + "\n", encoding="utf-8")
for split, report in chime6_report.items():
    print(f"{split}: {report['clip_count']} 个片段 / {report['hours']:.2f} h")
    for row in report["results"]:
        print(f"  阈值 {row['cutoff']:.2f}: {row['false_accepts_per_hour']:.2f} 次/小时"
              f"（{row['false_accepts']:.1f} 次，涉及 {row['clips_with_any_trigger']} 个片段）")
    for clip in report["loudest_clips"][:3]:
        print(f"    峰值 {clip['peak_probability']:.3f}  {clip['session']} "
              f"{clip['room']} {clip['array']} {clip['path'].rsplit('_', 1)[-1]}")
print(f"报告：{report_path}")


In [42]:
# 打包训练结果和数据来源记录。
import shutil
import subprocess
import tempfile

marker = hey_hi_audio_dir / "generation.json"
marker_base = repo_path if marker.is_relative_to(repo_path) else hey_hi_audio_dir.parent
hey_hi_archive_inputs = [
    str((hey_hi_features_dir / "feature_sources.json").relative_to(repo_path)),
    "-C", str(marker_base), str(marker.relative_to(marker_base)),
]

# CHiME-6 的来源文件放进独立前缀，避免与 licensed_data 的 prepared/ 同名。
with tempfile.TemporaryDirectory() as staging:
    provenance = Path(staging) / "chime6" / "prepared"
    provenance.mkdir(parents=True)
    for name in ("manifest.json", "manifest.csv", "ATTRIBUTION.md"):
        shutil.copy2(chime6_dir / "prepared" / name, provenance / name)
    shutil.copytree(chime6_dir / "prepared" / "sources", provenance / "sources")
    subprocess.run([
        "tar", "-czf", str(archive_path),
        "-C", str(repo_path),
        str(train_dir.relative_to(repo_path)),
        str(config_path.relative_to(repo_path)),
        str((features_dir / "feature_sources.json").relative_to(repo_path)),
        str((sample_dir / "generation.json").relative_to(repo_path)),
        *hey_hi_archive_inputs,
        str((chime6_features_dir / "feature_sources.json").relative_to(repo_path)),
        "-C", str(staging),
        "chime6/prepared/manifest.json", "chime6/prepared/manifest.csv",
        "chime6/prepared/ATTRIBUTION.md", "chime6/prepared/sources",
        "-C", str(data_dir),
        "prepared/manifest.json", "prepared/manifest.csv", "prepared/ATTRIBUTION.md",
        "prepared/excluded_fsd.csv", "prepared/sources",
    ], check=True)
print(f"已创建：{archive_path}")


已创建：/content/hi_fondcircle_clean100_v1_hey_hi_run.tar.gz
